# Delete VQA rows where `triples_used` is empty

Notebook này sẽ:

1. Kết nối Supabase bằng biến môi trường trong `.env`
2. Duyệt toàn bộ bảng `vqa` theo từng page
3. Lọc các row có `triples_used` rỗng (`[]`)
4. Lưu backup ra file JSON/CSV
5. Xóa các row đó theo batch
6. Kiểm tra lại sau khi xóa

> Gợi ý: chạy lần lượt từng cell từ trên xuống.  
> Với cell delete, hãy xem trước count và backup rồi mới chạy.


In [1]:
from pathlib import Path
import os
import json

import pandas as pd
from dotenv import load_dotenv

# Nếu notebook nằm trong project root thì đoạn này thường là đủ.
# Nếu cần, bạn có thể đổi PROJECT_ROOT cho đúng thư mục repo.
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError("Thiếu SUPABASE_URL hoặc SUPABASE_KEY trong file .env")

print("Loaded .env from:", PROJECT_ROOT / ".env")
print("SUPABASE_URL:", SUPABASE_URL[:40] + "..." if len(SUPABASE_URL) > 40 else SUPABASE_URL)
print("SUPABASE_KEY loaded:", bool(SUPABASE_KEY))


Loaded .env from: d:\Code\HCMUS\NNHTK\ViFoodKG\.env
SUPABASE_URL: https://cvdoasxazyruytejluvv.supabase.co
SUPABASE_KEY loaded: True


In [2]:
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
supabase


In [3]:
PAGE_SIZE = 1000

def norm_text(value):
    if value is None:
        return ""
    return str(value).strip()

def is_empty_triples(value):
    """Trả True nếu triples_used là [] hoặc tương đương rỗng."""
    if value is None:
        return True
    if isinstance(value, list):
        return len(value) == 0
    if isinstance(value, str):
        s = value.strip()
        return s in {"", "[]"}
    return False

def fetch_all_vqa_rows_for_empty_triples(client, page_size=PAGE_SIZE):
    rows = []
    start = 0

    while True:
        resp = (
            client.table("vqa")
            .select("vqa_id, image_id, qtype, question, triples_used, created_at, updated_at")
            .order("vqa_id")
            .range(start, start + page_size - 1)
            .execute()
        )
        batch = resp.data or []
        if not batch:
            break

        rows.extend(batch)
        print(f"Fetched {len(rows)} rows so far...")

        if len(batch) < page_size:
            break
        start += page_size

    return rows


In [4]:
all_rows = fetch_all_vqa_rows_for_empty_triples(supabase)
print("Total rows in vqa:", len(all_rows))

empty_rows = [row for row in all_rows if is_empty_triples(row.get("triples_used"))]
print("Rows with empty triples_used:", len(empty_rows))


Fetched 1000 rows so far...
Fetched 2000 rows so far...
Fetched 3000 rows so far...
Fetched 4000 rows so far...
Fetched 5000 rows so far...
Fetched 6000 rows so far...
Fetched 7000 rows so far...
Fetched 8000 rows so far...
Fetched 9000 rows so far...
Fetched 9290 rows so far...
Total rows in vqa: 9290
Rows with empty triples_used: 262


In [5]:
df_empty = pd.DataFrame(empty_rows)
df_empty.head(20)


,vqa_id,image_id,qtype,question,triples_used,created_at,updated_at
0,11141,image000013,dietary_restrictions,Dựa vào vị trí và thành phần của các món ăn tr...,[],2026-03-22T06:16:19.629515+00:00,2026-03-22T06:16:19.629515+00:00
1,11142,image000018,dietary_restrictions,"Trong hình ảnh này, món ăn chính có màu nâu sẫ...",[],2026-03-22T06:16:19.629515+00:00,2026-03-22T06:16:19.629515+00:00
2,11147,image000027,dietary_restrictions,Nếu một người đang áp dụng chế độ ăn thuần thự...,[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
3,11150,image000035,dietary_restrictions,Nếu một người đang thực hiện chế độ ăn thuần t...,[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
4,11151,image000036,dietary_restrictions,"Trong khay thức ăn này, nếu một người ăn chay ...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
5,11154,image000058,dietary_restrictions,Nếu một người đang áp dụng chế độ ăn thuần thự...,[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
6,11156,image000072,dietary_restrictions,"Xét các món ăn trên bàn, nếu một người thực hi...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
7,11159,image000079,dietary_restrictions,"Xét các món ăn trên khăn trải bàn caro, nếu mộ...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
8,11160,image000080,dietary_restrictions,"Trong các hàng món ăn được bày biện, nếu một n...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
9,11164,image000088,dietary_restrictions,"Xét về chế độ ăn thuần thực vật, liệu sự kết h...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00


In [6]:
if not df_empty.empty:
    display(
        df_empty.groupby("qtype", dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["count", "qtype"], ascending=[False, True])
    )
else:
    print("Không có row nào có triples_used rỗng.")


,qtype,count
0,dietary_restrictions,262


In [7]:
backup_dir = PROJECT_ROOT / "data" / "debug_outputs" / "empty_triples_cleanup"
backup_dir.mkdir(parents=True, exist_ok=True)

backup_json = backup_dir / "vqa_rows_with_empty_triples.json"
backup_csv = backup_dir / "vqa_rows_with_empty_triples.csv"

with open(backup_json, "w", encoding="utf-8") as f:
    json.dump(empty_rows, f, ensure_ascii=False, indent=2)

df_empty.to_csv(backup_csv, index=False, encoding="utf-8-sig")

print("Saved backup JSON:", backup_json)
print("Saved backup CSV :", backup_csv)


Saved backup JSON: d:\Code\HCMUS\NNHTK\ViFoodKG\data\debug_outputs\empty_triples_cleanup\vqa_rows_with_empty_triples.json
Saved backup CSV : d:\Code\HCMUS\NNHTK\ViFoodKG\data\debug_outputs\empty_triples_cleanup\vqa_rows_with_empty_triples.csv


## Delete rows

Cell dưới đây sẽ xóa theo `vqa_id` từng batch nhỏ.

Mặc định biến `EXECUTE_DELETE = False` để tránh xóa nhầm.  
Sau khi bạn đã kiểm tra count và backup, hãy đổi thành `True` rồi chạy cell.


In [8]:
DELETE_BATCH_SIZE = 200
EXECUTE_DELETE = False

vqa_ids_to_delete = [row["vqa_id"] for row in empty_rows if row.get("vqa_id") is not None]
print("Prepared vqa_ids to delete:", len(vqa_ids_to_delete))

if not EXECUTE_DELETE:
    print("Delete is disabled. Set EXECUTE_DELETE = True rồi chạy lại cell này để xóa.")
else:
    deleted = 0
    for i in range(0, len(vqa_ids_to_delete), DELETE_BATCH_SIZE):
        batch_ids = vqa_ids_to_delete[i:i + DELETE_BATCH_SIZE]
        (
            supabase.table("vqa")
            .delete()
            .in_("vqa_id", batch_ids)
            .execute()
        )
        deleted += len(batch_ids)
        print(f"Deleted {deleted}/{len(vqa_ids_to_delete)} rows")

    print("Delete completed.")


Prepared vqa_ids to delete: 262
Delete is disabled. Set EXECUTE_DELETE = True rồi chạy lại cell này để xóa.


In [9]:
# Verify lại sau khi xóa
remaining_rows = fetch_all_vqa_rows_for_empty_triples(supabase)
remaining_empty_rows = [row for row in remaining_rows if is_empty_triples(row.get("triples_used"))]

print("Remaining rows with empty triples_used:", len(remaining_empty_rows))

pd.DataFrame(remaining_empty_rows).head(20)


Fetched 1000 rows so far...
Fetched 2000 rows so far...
Fetched 3000 rows so far...
Fetched 4000 rows so far...
Fetched 5000 rows so far...
Fetched 6000 rows so far...
Fetched 7000 rows so far...
Fetched 8000 rows so far...
Fetched 9000 rows so far...
Fetched 9290 rows so far...
Remaining rows with empty triples_used: 262


,vqa_id,image_id,qtype,question,triples_used,created_at,updated_at
0,11141,image000013,dietary_restrictions,Dựa vào vị trí và thành phần của các món ăn tr...,[],2026-03-22T06:16:19.629515+00:00,2026-03-22T06:16:19.629515+00:00
1,11142,image000018,dietary_restrictions,"Trong hình ảnh này, món ăn chính có màu nâu sẫ...",[],2026-03-22T06:16:19.629515+00:00,2026-03-22T06:16:19.629515+00:00
2,11147,image000027,dietary_restrictions,Nếu một người đang áp dụng chế độ ăn thuần thự...,[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
3,11150,image000035,dietary_restrictions,Nếu một người đang thực hiện chế độ ăn thuần t...,[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
4,11151,image000036,dietary_restrictions,"Trong khay thức ăn này, nếu một người ăn chay ...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
5,11154,image000058,dietary_restrictions,Nếu một người đang áp dụng chế độ ăn thuần thự...,[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
6,11156,image000072,dietary_restrictions,"Xét các món ăn trên bàn, nếu một người thực hi...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
7,11159,image000079,dietary_restrictions,"Xét các món ăn trên khăn trải bàn caro, nếu mộ...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
8,11160,image000080,dietary_restrictions,"Trong các hàng món ăn được bày biện, nếu một n...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
9,11164,image000088,dietary_restrictions,"Xét về chế độ ăn thuần thực vật, liệu sự kết h...",[],2026-03-22T08:29:20.114696+00:00,2026-03-22T08:29:20.114696+00:00
